# Phase 2 — CLV Modelling (BG/NBD + Gamma-Gamma)

Fits probabilistic CLV models and produces 90-day and 365-day scores for every repeat customer.

**Sections**
1. Setup & data preparation
2. BG/NBD model fitting & diagnostics
3. Gamma-Gamma assumption check & fitting
4. CLV scoring & distributions
5. CLV vs actual revenue
6. CLV segment analysis
7. Summary

## 1. Setup & Data Preparation

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.plotting import plot_frequency_recency_matrix, plot_probability_alive_matrix
from models.data_pipeline import (
    load_raw_data, extract_signal_before_cleaning,
    clean_data, build_master_customer_table
)
from models.clv_model import (
    prepare_clv_data, fit_bgnbd, predict_purchases,
    check_gg_assumption, fit_gamma_gamma, score_clv,
    save_models, build_clv_scores
)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
df_raw = load_raw_data()
return_features, cancel_features = extract_signal_before_cleaning(df_raw)
df_customers, _ = clean_data(df_raw)
master = build_master_customer_table(df_customers, return_features, cancel_features)
clv_df = prepare_clv_data(master)
print(clv_df[['frequency_repeat', 'recency_bgnbd', 'T_bgnbd', 'monetary']].describe().round(2))

## 2. BG/NBD Model Fitting & Diagnostics

In [ ]:
bgf = fit_bgnbd(clv_df)
predicted_purchases_90d, prob_alive = predict_purchases(bgf, clv_df)

---
### Plot 1 — Frequency × Recency Matrix
**Business question: For a given recency and frequency, how many purchases does the model expect in the next 90 days?**

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
plot_frequency_recency_matrix(bgf, T=90, ax=ax)
ax.set_title('BG/NBD — Expected Purchases in Next 90 Days')
plt.tight_layout()
plt.show()

---
### Plot 2 — Probability Alive Matrix
**Business question: What is the probability that a customer with a given recency and frequency is still active?**

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
plot_probability_alive_matrix(bgf, ax=ax)
ax.set_title('BG/NBD — Probability Customer is Still Alive')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(prob_alive, bins=60, color='steelblue', edgecolor='white')
ax.axvline(0.5, color='red', linestyle='--', linewidth=1.5, label='50% threshold')
ax.set_title('Distribution of P(Alive) Across Repeat Customers')
ax.set_xlabel('Probability Alive')
ax.legend()
pct_alive = (prob_alive > 0.5).mean() * 100
ax.text(0.55, ax.get_ylim()[1] * 0.85, f'{pct_alive:.1f}% of customers\nhave P(alive) > 0.5', fontsize=11)
plt.tight_layout()
plt.show()

## 3. Gamma-Gamma Assumption Check & Fitting

---
### Assumption Check — Frequency vs Monetary Independence
**Business question: Is the Gamma-Gamma model's independence assumption satisfied for this dataset?**

In [ ]:
corr = check_gg_assumption(clv_df)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
freq_clip = clv_df['frequency_repeat'].clip(upper=clv_df['frequency_repeat'].quantile(0.97))
mon_clip = clv_df['monetary'].clip(upper=clv_df['monetary'].quantile(0.97))
ax.scatter(freq_clip, mon_clip, alpha=0.2, s=8, color='steelblue')
ax.set_xlabel('Frequency (repeat purchases)')
ax.set_ylabel('Monetary (mean order £)')
ax.set_title(f'Frequency vs Monetary  (Pearson r = {corr:.3f})')
if abs(corr) > 0.3:
    ax.text(0.05, 0.92, 'WARNING: |r| > 0.3 — independence assumption may be violated',
            transform=ax.transAxes, color='red', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
ggf = fit_gamma_gamma(clv_df)

## 4. CLV Scoring & Distributions

In [ ]:
expected_avg_order_value, clv_90d, clv_365d = score_clv(bgf, ggf, clv_df)
scores = clv_df[['customer_id', 'frequency_repeat', 'monetary', 'total_revenue']].copy()
scores['prob_alive'] = prob_alive.values
scores['predicted_purchases_90d'] = predicted_purchases_90d.values
scores['expected_avg_order_value'] = expected_avg_order_value.values
scores['clv_90d'] = clv_90d.values
scores['clv_365d'] = clv_365d.values
print(scores[['clv_90d', 'clv_365d', 'prob_alive', 'expected_avg_order_value']].describe().round(2))

---
### Plot 3 — CLV Distribution (Log Scale)
**Business question: What does the distribution of predicted CLV look like — is it heavily right-skewed?**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
clv_90_pos = scores[scores['clv_90d'] > 0]['clv_90d']
axes[0].hist(clv_90_pos, bins=80, color='steelblue', edgecolor='none', alpha=0.85)
axes[0].set_yscale('log')
axes[0].set_title('CLV 90-Day Distribution (log y-axis)')
axes[0].set_xlabel('Predicted CLV — 90 days (£)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
clv_365_pos = scores[scores['clv_365d'] > 0]['clv_365d']
axes[1].hist(clv_365_pos, bins=80, color='seagreen', edgecolor='none', alpha=0.85)
axes[1].set_yscale('log')
axes[1].set_title('CLV 365-Day Distribution (log y-axis)')
axes[1].set_xlabel('Predicted CLV — 365 days (£)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.suptitle('Predicted CLV Distributions — Both Horizons', fontsize=13)
plt.tight_layout()
plt.show()

## 5. CLV vs Actual Revenue

---
### Plot 4 — Predicted CLV vs Actual Total Revenue
**Business question: How well does the predicted CLV correlate with each customer's observed revenue?**

In [ ]:
scores_plot = scores.copy()
scores_plot['clv_365d_clip'] = scores_plot['clv_365d'].clip(upper=scores_plot['clv_365d'].quantile(0.97))
scores_plot['total_revenue_clip'] = scores_plot['total_revenue'].clip(upper=scores_plot['total_revenue'].quantile(0.97))
fig = px.scatter(
    scores_plot.sample(min(3000, len(scores_plot)), random_state=42),
    x='clv_365d_clip', y='total_revenue_clip',
    color='prob_alive', color_continuous_scale='RdYlGn',
    opacity=0.5, trendline='ols',
    labels={
        'clv_365d_clip': 'Predicted CLV 365d (£)',
        'total_revenue_clip': 'Actual Total Revenue (£)',
        'prob_alive': 'P(Alive)'
    },
    title='Predicted CLV (365d) vs Actual Total Revenue'
)
fig.show()
corr_clv = scores['clv_365d'].corr(scores['total_revenue'])
print(f'Pearson correlation (CLV 365d vs actual revenue): {corr_clv:.3f}')

## 6. CLV Segment Analysis

---
### Plot 5 — CLV Tier Breakdown
**Business question: How many customers fall into each CLV tier and what is their combined predicted revenue?**

In [ ]:
scores['clv_tier'] = pd.qcut(
    scores['clv_365d'],
    q=[0, 0.25, 0.5, 0.75, 1.0],
    labels=['Low', 'Mid', 'High', 'Top']
)

tier_summary = (
    scores.groupby('clv_tier', observed=True)
    .agg(
        customer_count=('customer_id', 'count'),
        median_clv_365d=('clv_365d', 'median'),
        total_predicted_revenue=('clv_365d', 'sum'),
        median_prob_alive=('prob_alive', 'median'),
    )
    .reset_index()
)
print(tier_summary.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(tier_summary['clv_tier'].astype(str), tier_summary['customer_count'],
            color=['#d9534f', '#f0ad4e', '#5bc0de', '#5cb85c'])
axes[0].set_title('Customer Count by CLV Tier')
axes[0].set_xlabel('CLV Tier')
axes[0].set_ylabel('Customers')
for i, v in enumerate(tier_summary['customer_count']):
    axes[0].text(i, v + v * 0.01, f'{v:,}', ha='center', fontsize=10)

axes[1].bar(tier_summary['clv_tier'].astype(str), tier_summary['total_predicted_revenue'],
            color=['#d9534f', '#f0ad4e', '#5bc0de', '#5cb85c'])
axes[1].set_title('Total Predicted Revenue by CLV Tier (365d)')
axes[1].set_xlabel('CLV Tier')
axes[1].set_ylabel('Revenue (£)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e6:.1f}M'))

plt.suptitle('CLV Tier Breakdown', fontsize=13)
plt.tight_layout()
plt.show()

---
### Plot 6 — P(Alive) by CLV Tier
**Business question: Are high-CLV customers also more likely to still be active?**

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
tier_order = ['Low', 'Mid', 'High', 'Top']
scores.boxplot(
    column='prob_alive', by='clv_tier',
    ax=ax, patch_artist=True
)
ax.set_title('P(Alive) Distribution by CLV Tier')
ax.set_xlabel('CLV Tier')
ax.set_ylabel('Probability Alive')
plt.suptitle('')
plt.tight_layout()
plt.show()

## Saving Models & Scores

In [ ]:
save_models(bgf, ggf)

from pathlib import Path
scores[['customer_id', 'frequency_repeat', 'prob_alive',
        'predicted_purchases_90d', 'expected_avg_order_value',
        'clv_90d', 'clv_365d']].to_csv(
    Path('..') / 'data' / 'processed' / 'clv_scores.csv', index=False
)
print('saved models/bgf.pkl, models/ggf.pkl, data/processed/clv_scores.csv')

## 7. Summary

In [ ]:
print('=== Phase 2 Complete ===')
print(f'repeat customers scored  : {len(scores):,}')
print(f'median CLV 90d           : £{scores["clv_90d"].median():.2f}')
print(f'median CLV 365d          : £{scores["clv_365d"].median():.2f}')
print(f'mean   CLV 365d          : £{scores["clv_365d"].mean():.2f}')
print(f'% customers P(alive)>0.5 : {(scores["prob_alive"] > 0.5).mean() * 100:.1f}%')
print(f'\nCLV tier distribution:')
print(tier_summary[['clv_tier', 'customer_count', 'median_clv_365d']].to_string(index=False))